# IDP Hopfield — corrected

Fixes vs. the original notebook:
1. `alpha = (1/N) * xi^T u`, not `xi^T u`  (was N times too large -> no selectivity, numerical blowup)
2. Memories are **centered** before building W  (MNIST digits share a huge background -> not orthogonal)
3. No `fill_diagonal(w, 0)`  (not part of the continuous IDP model)
4. Input **gain** so the dominant alpha clears the existence threshold `1/beta`
5. Random initial condition (that IS the IDP claim), smaller `dt`, convergence check
6. Sign-invariant classification (equilibria come in symmetric pairs +/- gamma*xi)


In [ ]:
"""
IDP Hopfield — corrected implementation (Betteti et al. 2024/2025).

Model:  xdot = -x + W(alpha) Psi(x),   W(alpha) = (1/N) sum_mu alpha_mu xi^mu xi^mu^T
        alpha_mu = (1/N) xi^mu^T u          <-- the (1/N) is essential
        Psi(x) = tanh(beta x)
        existence threshold: alpha_mu > 1/beta   (paper uses beta=1 -> alpha>1)
"""
import numpy as np
import matplotlib.pyplot as plt

N = 784
BETA = 5.0

In [ ]:
# ---------------------------------------------------------------- data
def load_memories(path="basic-memory-ds/mnist_processed.csv", n_mem=10, seed=0):
    """Return Xi (P, N) in {-1,+1}. Falls back to synthetic MNIST-like data."""
    try:
        import pandas as pd
        ds = pd.read_csv(path)
        X = ds.to_numpy(dtype=float)
        print(f"loaded {X.shape}, value range [{X.min()}, {X.max()}]")
        if X.shape[1] == N + 1:                       # label column present
            X = X[:, 1:]
        assert X.shape[1] == N, f"expected {N} pixel columns, got {X.shape[1]}"
        assert np.isfinite(X).all(), "NaNs/infs in the CSV"
        if X.max() > 1.5:                             # 0..255 -> bipolar
            X = X / 255.0
        Xi = np.where(X >= 0.5 * (X.max() + X.min()), 1.0, -1.0)
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(Xi), min(n_mem, len(Xi)), replace=False)
        return Xi[idx]
    except (FileNotFoundError, ImportError) as e:
        print(f"[{e}] -> synthetic MNIST-like memories")
        rng = np.random.default_rng(seed)
        Xi = np.tile(np.full(N, -1.0), (n_mem, 1))
        for m in range(n_mem):
            Xi[m, rng.choice(N, 150, replace=False)] = 1.0
        return Xi

In [ ]:
def distort(v, pct, rng):
    v = np.array(v, dtype=float).copy()
    k = round(pct * v.size)
    idx = rng.choice(v.size, k, replace=False)
    v[idx] = rng.uniform(-1, 1, k)
    return v

In [ ]:
# ------------------------------------------------------- the IDP model
def psi(x, beta=BETA):
    return np.tanh(beta * x)

In [ ]:
def saliency(Xi_c, u_c, gain=1.0):
    """alpha_mu = gain * (1/N) <xi^mu, u>.  O(1), comparable to threshold 1/beta."""
    return gain * (Xi_c @ u_c) / Xi_c.shape[1]

In [ ]:
def synaptic(Xi_c, alpha):
    """W(alpha) = (1/N) Xi^T diag(alpha) Xi.  No diagonal zeroing (not in the model)."""
    return (Xi_c.T * alpha) @ Xi_c / Xi_c.shape[1]

In [ ]:
def relax(W, x0, dt=0.01, t_max=20.0, tol=1e-8, beta=BETA):
    """Forward Euler with a convergence check. dt must satisfy dt < 1/(beta*||W||)."""
    x = np.asarray(x0, dtype=float).copy()
    steps = int(t_max / dt)
    for k in range(steps):
        dx = -x + W @ psi(x, beta)
        x = x + dt * dx
        if not np.isfinite(x).all():
            raise FloatingPointError(f"diverged at step {k}: dt too large or W too big")
        if np.max(np.abs(dx)) < tol:
            break
    return x

In [ ]:
def cosines(Xi_c, x):
    nx = np.linalg.norm(x)
    if nx < 1e-9:
        return np.zeros(len(Xi_c))
    return (Xi_c @ x) / (np.linalg.norm(Xi_c, axis=1) * nx)

In [ ]:
def classify(Xi_c, x, thresh=0.5, min_norm=1e-2):
    """Sign-invariant: IDP equilibria come in symmetric pairs +-gamma*xi.
    A collapsed state (x ~ 0) is the confusion state, not a retrieval."""
    c = cosines(Xi_c, x)
    if np.linalg.norm(x) < min_norm:
        return None, c
    j = int(np.argmax(np.abs(c)))
    return (j, c) if abs(c[j]) > thresh else (None, c)

In [ ]:
# ------------------------------------------------------------- driver
def run(pct=0.4, gain=3.0, seed=0, target=0, random_init=True, verbose=True):
    rng = np.random.default_rng(seed)
    Xi = load_memories(seed=seed) if "Xi_cache" not in run.__dict__ else run.Xi_cache
    run.Xi_cache = Xi

    mean = Xi.mean(axis=0)          # <-- kills the shared-background direction
    Xi_c = Xi - mean

    clean = Xi[target]
    cue = distort(clean, pct, rng)
    u_c = cue - mean

    alpha = saliency(Xi_c, u_c, gain)
    W = synaptic(Xi_c, alpha)

    x0 = rng.normal(0, 0.01, N) if random_init else u_c
    x = relax(W, x0)

    pred, c = classify(Xi_c, x)
    if verbose:
        print(f"pct={pct} gain={gain} random_init={random_init}")
        print(f"  alpha: max={alpha.max():.3f} (target={alpha[target]:.3f}), "
              f"2nd={np.sort(alpha)[-2]:.3f}, threshold=1/beta={1/BETA:.3f}")
        print(f"  ||x||={np.linalg.norm(x):.3f}  "
              f"{'CONFUSION STATE (origin)' if np.linalg.norm(x) < 1e-2 else ''}")
        print(f"  cos to target={c[target]:+.3f}, "
              f"max |cos| to others={np.max(np.abs(np.delete(c, target))):.3f}")
        print(f"  predicted={pred}  -> {'OK' if pred == target else 'FAIL'}\n")
    return Xi, Xi_c, clean, cue, x, alpha, mean

In [ ]:
def show(vec, title, mean=None, flip=False):
    v = np.array(vec, float)
    if mean is not None:
        v = v + mean
    if flip:
        v = -v
    plt.imshow(np.where(v >= 0, 1.0, 0.0).reshape(28, 28), cmap="gray")
    plt.title(title, fontsize=13, fontweight="bold", pad=12)
    plt.axis("off")
    plt.show()

In [ ]:
print("=== correct settings, random initial condition (the IDP claim) ===")
run(pct=0.4, gain=3.0, random_init=True)

print("=== gain too low -> alpha below existence threshold -> confusion state ===")
run(pct=0.7, gain=1.0, random_init=True)

print("=== NO centering (the MNIST failure mode) ===")
rng = np.random.default_rng(0)
Xi = load_memories(seed=0)
cue = distort(Xi[0], 0.4, rng)
a = saliency(Xi, cue, 3.0)
x = relax(synaptic(Xi, a), rng.normal(0, 0.01, N))
c = cosines(Xi, x)
print(f"  alpha: {np.round(a, 3)}")
print(f"  cos to every memory: {np.round(c, 3)}  <- mixed state, no winner\n")

print("=== old alpha scaling (no 1/N) ===")
a_bad = Xi @ cue
print(f"  alpha ranges up to {a_bad.max():.0f}; threshold is {1/BETA:.2f}. "
      f"Every memory is {a_bad.min()/(1/BETA):.0f}-{a_bad.max()/(1/BETA):.0f}x "
      f"over threshold -> no selectivity.")